In [1]:
from src.utils import get_data_env
from src.UPDATED_dataloading import EpiConfig, DataOrchestrator
from src.UPDATED_models import PersistenceModel, NodeRFModel
from src.UPDATED_dataloading import GraphDataLoaderManager, ShallowDataLoaderManager

from src.UPDATED_models import GATv2Model

disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        = '2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False

horizon_size    = 2
horizon_leadtime= 3
sequence_length = 1
lags            = 1

config = EpiConfig(
    disease             = 'influenza',
    data_env_dir        = get_data_env(),
    date_range          = (min_date, max_date),
    horizon_size        = horizon_size,
    sequence_length     = sequence_length,
    horizon_leadtime    = horizon_leadtime,
    lags                = lags,
    nuts_level          = nuts_level,
    log_transform       = True,
    split_berlin        = split_berlin,
    include_population  = False,
    split_trainval      = split_trainval, 
    split_valtest       = split_valtest
    )    

dataorchestrator    = (DataOrchestrator(config).build())

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
n_epochs        = 250
lr              = 0.0001
min_delta       = 0.0001
loss            = 'mse'

global_hparams = {
    "lr"                : lr,
    "n_epochs"          : n_epochs,
    "scheduler"         : 'plateau',
    "scheduler_kwargs"  :   {'mode': 'min', 'factor': 0.5, 'patience': 7},
    'min_delta'         : min_delta,
    'loss'              : loss,
    'patience'          : 20,
    }

shallowdata        = ShallowDataLoaderManager(dataorchestrator).construct_dataloaders()
graphdataloader_id  = GraphDataLoaderManager(dataorchestrator).retrieve_graph('identity_graph').construct_dataloaders()
graphdataloader_bn  = GraphDataLoaderManager(dataorchestrator).retrieve_graph('boolean_neighbors_self').construct_dataloaders()
graphdataloader_gr1 = GraphDataLoaderManager(dataorchestrator).retrieve_graph('gravity1').construct_dataloaders()
graphdataloader_cm1 = GraphDataLoaderManager(dataorchestrator).retrieve_graph('commuter1').construct_dataloaders()

persistence = PersistenceModel(shallowdata)
persistence.forecast('test')

rf = NodeRFModel(shallowdata)
rf.set_global_hparams()
rf.set_model_hparams(n_estimators=100)
rf.train(verbose=2)
rf.forecast('test')

model1 = GATv2Model(graphdataloader_id, name = 'identity_graph')
model1.set_model_hparams()
model1.set_global_hparams(**global_hparams)
model1.train(verbose = 2)
model1.forecast('test')
model1.show_forecasts(26)

model2 = GATv2Model(graphdataloader_bn, name = 'boolean neighbors')
model2.set_model_hparams()
model2.set_global_hparams(**global_hparams)
model2.train(verbose = 2)
model2.forecast('test')
model2.show_forecasts(26)

model3 = GATv2Model(graphdataloader_gr1, name = 'gravity 1')
model3.set_model_hparams()
model3.set_global_hparams(**global_hparams)
model3.train(verbose = 2)
model3.forecast('test')
model3.show_forecasts(26)

model4 = GATv2Model(graphdataloader_cm1, name = 'commuter 1')
model4.set_model_hparams()
model4.set_global_hparams(**global_hparams)
model4.train(verbose = 2)
model4.forecast('test')
model4.show_forecasts(26)

Training loss for horizon 0: RMSE = 0.6200, MAE = 0.2463
Validation loss for horizon 0: RMSE = 0.8374, MAE = 0.3672
Training loss for horizon 1: RMSE = 0.6928, MAE = 0.2795
Validation loss for horizon 1: RMSE = 0.9616, MAE = 0.4173

==                identity_graph                ==
Dataloader Snapshot: GraphData(x=(400, 3, 1), y=(400, 2), edge_index=(2, 400), edge_weight=(400,))
Epoch 1 train loss: 0.7748, val loss: 1.2917 ✓ (new best)
Epoch 2 train loss: 0.7260, val loss: 1.1623 ✓ (new best)
Epoch 3 train loss: 0.7014, val loss: 1.0708 ✓ (new best)
Epoch 4 train loss: 0.6817, val loss: 0.9935 ✓ (new best)
Epoch 5 train loss: 0.6713, val loss: 0.9410 ✓ (new best)
Epoch 6 train loss: 0.6599, val loss: 0.9000 ✓ (new best)
Epoch 7 train loss: 0.6563, val loss: 0.8843 ✓ (new best)
Epoch 8 train loss: 0.6519, val loss: 0.8480 ✓ (new best)
Epoch 9 train loss: 0.6419, val loss: 0.8421 ✓ (new best)
Epoch 10 train loss: 0.6422, val loss: 0.8140 ✓ (new best)
Epoch 11 train loss: 0.6334, val los

In [ ]:
from src.UPDATED_evaluation.evaluator import Evaluator
evaluation = Evaluator([persistence, rf, model1, model2, model3, model4])

evaluation.add_evaluation(0, 'test')
evaluation.add_evaluation(1, 'test')

evaluation.plotter.plot_metric('ccc', 'horizon_0', 'test', 'box', 26)
evaluation.plotter.plot_metric('ccc', 'horizon_1', 'test', 'box', 26)

AttributeError: 'GATv2Model' object has no attribute 'clean_name'